![interpreto_banner](../assets/img/interpreto_banner.png)

# Classification Concept-based Explanation Tutorial

Welcome to this tutorial, our will be to obtain concept-based explanations starting from the beginning.

For any precision, please refer to the [**Interpreto documentation**](https://for-sight-ai.github.io/interpreto/).

There are five key steps for concepts based explanations:

1. [**Split** your model in two parts](#split)
2. [Compute a dataset of **activations**](#activations)
3. [**Fit** a concept model on activations](#fit)
4. [**Interpret** the concept dimensions](#interpret)
5. [Find the globally **important** concepts](#important)

On which we add three bonus steps:

6. [**Class-wise** concepts and LLM label](#class-wise)
7. [**Locally** important concepts](#locally)
8. [**Evaluate** concept-based explanations](#evaluate)

*Author: Antonin Poché*

In [1]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. **Split** your model in two parts <a class="anchor" id="split"></a>

We choose a `DistilBERT` fine-tuned on the `AG-News` dataset and split it just before the classification head.

To split the model, we use the [`interpreto.ModelWithSplitPoints`](https://for-sight-ai.github.io/interpreto/api/concepts/model_with_split_points/) which wraps around the `transformers` model and allows the computation of activations at the specified `split_points`.

In [2]:
from transformers import AutoModelForSequenceClassification

from interpreto import ModelWithSplitPoints

model_with_split_points = ModelWithSplitPoints(
    model_or_repo_id="textattack/distilbert-base-uncased-ag-news",
    automodel=AutoModelForSequenceClassification,
    split_points=[5],  # split at the fifth layer
    device_map="cuda",
    batch_size=1024,
)

## 2. Compute a datasets of **activations** <a class="anchor" id="activations"></a>

We load the first 10000 documents of the `AG-News` train set.

Then we extract the activations of the [CLS] token of each document.

> **Common practice**
>
> In the literature, to train concepts for classification it is common to use the [CLS] just before the classification head.
>
> In fact, at this layer, it makes no sense to use other elements.

> **Warning**
>
> In this notebook, many things are specific to the use of the [CLS] token.

[`interpreto.ModelWithSplitPoints.get_activations()`](https://for-sight-ai.github.io/interpreto/api/concepts/model_with_split_points/#interpreto.ModelWithSplitPoints.get_activations)

In [3]:
from datasets import load_dataset

# load the AG-News dataset
dataset = load_dataset("fancyzhx/ag_news")
inputs = dataset["train"]["text"][:1000]  # TODO: increase, the more the better
classes_names = dataset["train"].features["label"].names

# Compute the [CLS] token activations
granularity = ModelWithSplitPoints.activation_granularities.CLS_TOKEN
activations = model_with_split_points.get_activations(
    inputs=inputs,
    activation_granularity=granularity,
    tqdm_bar=True,
    include_predicted_classes=True,
)

Computing activations: 100%|██████████| 1/1 [00:02<00:00,  2.65s/batch]


## 3. **Fit** a concept model on activations <a class="anchor" id="fit"></a>

With activations, we can train a concept model to find patterns (concepts).

The `concept_model` is an attribute of our concept explainer, similarly to the `model_with_split_points`. With these these two elements, we can go from inputs to concepts and from concepts to outputs.

In this tutorial, we use [`interpreto.concepts.ICAConcepts`](https://for-sight-ai.github.io/interpreto/api/concepts/methods/optim/#interpreto.concepts.ICAConcepts) built upon the ICA (Independent Component Analysis) dimension reduction algorithm.

There are at least 15 others concept model available in interpreto. do not hesitate to explore them.

> **Tip**
>
> `ICAConcepts` is a good first candidate for classification. It has no requirements and show good performances on most datasets.

In [4]:
from interpreto.concepts import ICAConcepts

# instantiate the concept explainer
concept_explainer = ICAConcepts(model_with_split_points, nb_concepts=50, device="cuda")

# fit the concept explainer on activations
concept_explainer.fit(activations)

## 4. **Interpret** the concept dimensions <a class="anchor" id="interpret"></a>

We have our concepts and the link between concepts and classes. But now, we need to make sense of these concepts.

In this case, we will use the [`interpreto.concepts.interpretations.TopKInputs`](https://for-sight-ai.github.io/interpreto/api/concepts/concepts_interpretations/#interpreto.concepts.interpretations.TopKInputs) to find the 8 words which activates the most our concepts.

> **Warning**
>
> If the `granularity` specified to the interpretation method is not the same as the one used for activations, the results will be wrong.

In [5]:
from interpreto.concepts.interpretations import TopKInputs

# instantiate the interpretation method with the concept explainer
topk_inputs_method = TopKInputs(
    concept_explainer=concept_explainer,
    k=5,
    activation_granularity=granularity,
    concept_encoding_batch_size=2**12,  # 4096
    use_unique_words=True,  # with the [CLS] token granularity, we are forced to use unique words
    unique_words_kwargs={
        "count_min_threshold": round(len(inputs) * 0.002),  # appear in at least 0.2% of the samples | increase if random words appear and decrease if some words appear too often
        "lemmatize": True,
        "words_to_ignore": [],  # include noise words and punctuation
    },
)

In [6]:
# call the interpretation methods on the inputs
# we cannot give the previously computed activations because `use_unique_words=True` creates samples with a single word inside
topk_words = topk_inputs_method.interpret(
    inputs=inputs,
    concepts_indices="all",
)

## 5. Find the globally **important** concepts <a class="anchor" id="important"></a>

We have concept directions, it means that our model has access to them, but not that it uses them.

It is the same when you train a model on tabular data, not all features are used.

In this step, we use the [`ConceptAutoEncoderExplainer.concept_output_gradients`](https://for-sight-ai.github.io/interpreto/api/concepts/methods/base/#interpreto.concepts.ConceptAutoEncoderExplainer.concept_output_gradient) to evaluate the importance of each concept with respect to the predicted classes.

> **Note**
>
> All unsupervised concept-based explainers in Interpreto inherit from [`ConceptAutoEncoderExplainer`](https://for-sight-ai.github.io/interpreto/api/concepts/methods/base/#interpreto.concepts.ConceptAutoEncoderExplainer).

> **Note 2**
>
> This step can be done prior to the interpretation, as the interpretation step can be compute heavy. Then specify using the `concept_indices` parameter.
> Only interpreting the important concepts can be wise. (Here we only have 50 concepts, so it does not matter.)

In [7]:
# estimate the importance of concepts for each class using the gradient
gradients = concept_explainer.concept_output_gradient(
    inputs=inputs,
    targets=None,  # None means all classes
    activation_granularity=granularity,
    concepts_x_gradients=True,  # the concept to output gradients are multiplied by the concepts values, this is common practice in the literature
    batch_size=64,
)

gradients = torch.stack(gradients, axis=1).squeeze()  # (num_classes, num_samples, num_concepts)
print(f"{gradients.shape=}")

Computing gradients: 100%|██████████| 16/16 [00:09<00:00,  1.76batches/s]

gradients.shape=torch.Size([4, 1000, 50])


We normalize the concepts importance to sum to 1 for each sample-class pair. 

In [8]:
import torch

# normalize the gradients
sample_class_importance_sum = gradients.abs().sum(dim=-1, keepdim=True)
normalized_gradients = gradients.abs() / sample_class_importance_sum  # (num_classes, num_samples, num_concepts)

# aggregate over samples
normalized_mean_gradients = normalized_gradients.mean(dim=1)  # (num_classes, num_concepts)

# for each class, sort the importance scores
order = torch.argsort(normalized_mean_gradients, descending=True)

for target in range(order.shape[0]):
    print(f"\n5 most important concepts for target {classes_names[target]}:")
    print("\t", [f"{order[target, i]}: {round(normalized_mean_gradients[target][order[target, i]].item(), 3)}" for i in range(5)] )


5 most important concepts for target World:
	 ['27: 0.091', '45: 0.078', '11: 0.07', '31: 0.064', '48: 0.041']

5 most important concepts for target Sports:
	 ['7: 0.111', '49: 0.092', '30: 0.087', '33: 0.069', '24: 0.062']

5 most important concepts for target Business:
	 ['13: 0.084', '28: 0.067', '19: 0.046', '7: 0.042', '37: 0.04']

5 most important concepts for target Sci/Tech:
	 ['7: 0.057', '49: 0.053', '13: 0.052', '27: 0.05', '30: 0.046']


Let's keep the 5 most important concepts for each class.

In [9]:
important_concept_indices = order[:, :5].flatten().unique().tolist()
print(f"\n{important_concept_indices=}")


important_concept_indices=[7, 11, 13, 19, 24, 27, 28, 30, 31, 33, 37, 45, 48, 49]


In [10]:
for target in range(order.shape[0]):
    print(f"\nMost important concepts for target {classes_names[target]}:")
    for i in range(5):
        words_importance = topk_words.get(order[target, i].item(), None)
        if words_importance is not None:
            print(f"\t{order[target, i]}: {list(words_importance.keys())}")
        else:
            print(f"\t{order[target, i]}: None")


Most important concepts for target World:
	27: ['serbia-montenegro', 'nato', 'liechtenstein', 'nikkei', 'ossetia']
	45: ['separatist', 'militia', 'militiaman', 'usatoday.com', 'inquirer']
	11: ['betting', 'saudi', 'gambler', 'shark', 'kidnapper']
	31: ['betting', 'fraud', 'holy', 'kmart', 'p.m.']
	48: ['afp', 'armed', 'hue', 'anarchist', 'naval']

Most important concepts for target Sports:
	7: ['phelps', '200-meter', 'batter', 'inning', 'homered']
	49: ['heat', '100-meter', 'fastest', '200-meter', '200m']
	30: ['200-meter', '100-meter', 'breaststroke', '400-meter', 'heat']
	33: ['phillies', 'mets', 'baltimore', 'sox', 'nl']
	24: ['stryker', 'nfl', 'armadillo', 'homer', 'autodesk']

Most important concepts for target Business:
	13: ['pharmacare', 'procurement', 'costly', 'sector', 'marketer']
	28: ['vodafone', 'antitrust', 'ipo.google.com', 'verizon', 'lenovo']
	19: [';', 'ott', 'atp', 'fcc', '3g']
	7: ['phelps', '200-meter', 'batter', 'inning', 'homered']
	37: ['eurozone', 'finance', 

> **The concepts are not interpretable, what do I do?**
>
> - Try to improve the concept-space:
>   - Increase the number of samples
>   - Try different concept-models and parameters
>   - Try to compute concepts class-wise see [next section](#class-wise)
>
> - Improve the interpretation of concepts:
>   - Play with the parameters
>   - Try [`LLMLabels`](https://for-sight-ai.github.io/interpreto/api/concepts/concepts_interpretations/#interpreto.concepts.interpretations.LLMLabels) see [next section](#class-wise)
>
> - Try to evaluate the concepts, to automatically find the best methods. Check this other tutorial: [TODO](TODO)
>
> - Never forget the **faithfulness-plausibility trade-off** of explanations

## 6. Better concepts with class-wise concepts and LLM labels <a class="anchor" id="class-wise"></a>

This section aims at improving the concepts learned by the model. We try three different approaches:

- Training class-wise concepts
- Using another concept model (TODO)
- Using LLM labels to interpret the concepts

When a single concept-space is defined for all classes, concepts tend to correspond to the classes themselves. In particular, when the concept-space is built upon on the latent space just before the classification head.

In this section, we will learn a concept space for each class separately. Thus, the class-wise concept explainers will only see examples from a single class (based on the predictions).

In [13]:
concept_explainers = {}
concept_interpretations = {}
concept_importances = {}

# iterate over classes
for target, class_name in enumerate(classes_names):
    print(f"\nComputing concepts for class: {class_name}")

    # extract activations for the class
    indices = (activations["predictions"] == target).nonzero(as_tuple=True)[0]
    class_wise_inputs = [inputs[i] for i in indices]
    class_wise_activations = {k: v[indices] for k, v in activations.items()}

    print(f"\tThere are {len(class_wise_inputs)} samples. Training ICA...")

    concept_explainer = ICAConcepts(model_with_split_points, nb_concepts=20, device="cuda")
    concept_explainer.fit(class_wise_activations)
    concept_explainers[target] = concept_explainer

    print("\tComputing concepts importance...")
    gradients = concept_explainer.concept_output_gradient(
        inputs=class_wise_inputs,
        targets=[target],
        activation_granularity=granularity,
        concepts_x_gradients=True,
        batch_size=64,
    )
    gradients = torch.stack(gradients, axis=1).squeeze()  # (num_samples, num_concepts)

    mean_gradients = gradients.abs().mean(dim=0)
    normalized_mean_gradients = mean_gradients / mean_gradients.sum(dim=0, keepdim=True)
    order = torch.argsort(normalized_mean_gradients, descending=True)
    concept_importances[target] = normalized_mean_gradients

    # important_concept_indices = order[:10].flatten().unique().tolist()

    print("\tInterpreting important concepts...")
    topk_inputs_method = LLMLabels(
        concept_explainer=concept_explainer,
        k=10,
        activation_granularity=granularity,
        concept_encoding_batch_size=2**12,  # 4096
        use_unique_words=True,
        unique_words_kwargs={
            "count_min_threshold": round(len(class_wise_inputs) * 0.002),  # appear in at least 0.2% of the samples
            "lemmatize": True,
            "words_to_ignore": [],
        },
    )

    topk_words = topk_inputs_method.interpret(
        inputs=class_wise_inputs,
        concepts_indices="all",  # all concepts
    )
    concept_interpretations[target] = topk_words

    print(f"\tMost important concepts for target {class_name}:")
    for concept_id in order[:10]:
        words_importance = topk_words.get(concept_id.item(), None)
        if words_importance is not None:
            print(f"\t\t{concept_id}: {list(words_importance.keys())}")
        else:
            print(f"\t\t{concept_id}: None")


Computing concepts for class: World
	There are 2456 samples. Training ICA...
	Computing concepts importance...


Computing gradients: 100%|██████████| 39/39 [00:14<00:00,  2.64batches/s]


	Interpreting important concepts...
	Most important concepts for target World:
		7: ['ivory', 'to\\protect', 'eclipsed', '\\', 'doe', 'venus', 'greenpeace', 'wasteland', 'scala', 'spar']
		19: ['rationed', 'per-share', 'deficit', 'finance', 'economy', 'inflation', 'wage', 'cash', 'surcharge', 'circulation']
		15: ['nuclear', 'smuggling', 'erupt', 'al-qaeda-linked', 'u.s.-backed', 'influenza', 'hostage-taking', 'u.s.-led', 'abducted', 'kidnapped']
		1: ['sanofi-aventis', 'pre-poll', 'twickenham', 'first-half', 'goodale', 'oakland', 'top-seeded', 'gasoline', 'pinch-hitter', 'al-beshir']
		8: ['olympic-record', 'six-nation', 'heptathlon', 'phelps', 'top-seeded', 'first-half', 'career-high', '200-meter', 'gold-medal', 'record-low']
		13: ["shi'ite\\militiamen", 'bustos', 'fla.', 'milosevic', 'loyalist', 'ex-paramilitary', 'rebel-held', 'blockaded', 'ex-rebel', 'cromwell']
		18: ['peshawar', 'liberate', 'blitz', 'hunger-striking', 'war-ravaged', 'al-qaeda', 'qaeda-linked', 'war-torn', 'mehd

/home/antonin.poche/interpreto/.venv/lib/python3.12/site-packages/sklearn/decomposition/_fastica.py:127: ConvergenceWarning: FastICA did not converge. Consider increasing tolerance or the maximum number of iterations.
  warnings.warn(


	Computing concepts importance...


...

	Interpreting important concepts...
	Most important concepts for target Sports:
		2: ['ioc', 'expo', 'nato', 'squash', 'delegation', 'badminton', 'judo', 'archery', 'weightlifting', 'fencing']
		6: ['olympics-henin', 'olympics-federer', 'seventh-seeded', 'west-leading', 'olympic-record', 'fifth-seeded', 'league-leading', 'pac-10', 'villarreal', 'second-seed']
		10: ['mamaroneck', 'calif.', 'barclays', 'cash', 'sbc', 'dividend', 'finance', 'pound', 'meares', 'cent']
		4: ['nba', 'pac-10', 'ronaldo', '36-hole', 'nhl', 'second-half', 'all-american', 'payton', 'pacer', 'goaltender']
		8: ['800-meter', '10,000-meter', '4,000-meter', 'middle-distance', '400-meter', 'iaaf', '3000m', '800m', '10,000m', '400m']
		17: ['pound', 'anastasia', 'jinx', 'baghdad', 'snatch', 'bustos', 'sept.', 'hee-sham', 'haile', 'saulnier']
		5: ['punter', 'shot-putters', 'five-shot', '12-yard', 'infielder', 'pound', 'four-shot', '5-yard', 'two-shot', 'dressage']
		9: ['infielder', 'nba', 'stats', 'shortstop', 'pres

Computing gradients: 100%|██████████| 38/38 [00:15<00:00,  2.52batches/s]


	Interpreting important concepts...
	Most important concepts for target Business:
		15: ['second-straight', '1st-half', '2nd-half', 'first-half', 'second-half', 'first-quarter', '2nd-quarter', 'second-quarter', 'jones/ap', 'td']
		6: ['overtime', '2nd-quarter', 'first-quarter', 'fourth-quarter', 'second-quarter', 'yukos', '1st-half', 'third-quarter', '2nd-half', 'atlanta']
		8: ['deli-style', 'multibillion-dollar', 'credit-card', 'euro', 'ex-enron', 'enron', 'nyse', 'kmart', 'paycheck', 'eurozone']
		5: ['profit-taking', 'multibillion-dollar', 'pay-per-view', '85/share', 'ticker=ltd.n', 'profitability', 'ticker=fdx.n', 'cnn/money', 'buffett', 'ticker=aci.n']
		7: ['petroleum', 'refinery', 'oil', 'coal', 'uranium', 'tobacco', 'oil-producing', 'iraqi', 'iraq', 'nuclear']
		17: ['home-improvement', 'calif.', 'calif', 'renovation', 'asbestos', 'housing', 'framingham', 'landmark', 'construction', 'property']
		19: ['oct.', 'aug.', 'rep.', 'toronto-dominion', 'nov.', 'macapagal-arroyo', 'sen

Computing gradients: 100%|██████████| 44/44 [00:17<00:00,  2.54batches/s]


	Interpreting important concepts...
	Most important concepts for target Sci/Tech:
		10: ['transistor', 'pentium', '802.11a', '802.11n', 'amd', 'hp-ux', 'microprocessor', 'semiconductor', '64-bit', 'supercomputer']
		14: ['calif.', 'terengganu', 'slate-colored', 'pent-up', 'anti-bush', 'sept.', 'iraq', 'islamabad', 'republican', 'methamphetamine']
		9: ['cyber-crime', 'e-government', 'p2pnet.net', 'siliconvalley.com', 'earthlink-hosted', 'pornography', 'hd-dvd', 'nanotechnology', 'macromedia', 'cybercafe']
		4: ['802.11a', '802.11n', 'forbes.com', 'flu', 'hd-dvd', 'u.k.', 'whale', 'siliconvalley.com', 'salesforce.com', 'guantanamo']
		5: ['second-quarter', 'fourth-quarter', 'nfl', 'rbot', 'crs-1', 'csco.o', 'doping', 'ap', 'discus', '15-in']
		2: ['gasoline', 'pound', 'midmarket', 'soybean', 'forbes.com', 'economy', 'gambling', 'fla.', 'cut-rate', 'cent']
		0: ['gasoline', 'soybean', 'cbs.mw', 'midmarket', 'flu', 'sewage', 'waste', 'outlook', '//ad.doubleclick.net/ad/idg.us.ifw.general/

## 6. **Locally** important concepts <a class="anchor" id="locally"></a>

In [11]:
test_examples = dataset["test"]["text"][:50]
test_labels = dataset["test"]["label"][:50]
test_preds = model_with_split_points.get_activations(
    inputs=test_examples,
    activation_granularity=granularity,
    tqdm_bar=True,
    include_predicted_classes=True,
)["predictions"]

for class_id, class_name in enumerate(classes_names):

    # extract example
    example_id = test_labels.index(class_id)
    example = test_examples[example_id]
    pred = test_preds[example_id].item()
    print(f"Local importance for example of class {class_name} (pred: {classes_names[pred]}):")
    print(f"Example: {example}")

    # compute local concepts importance for the class
    local_importance = concept_explainer.concept_output_gradient(
        inputs=[example],
        activation_granularity=granularity,
        concepts_x_gradients=True,
        tqdm_bar=False,
    )[0][class_id, 0]

    # normalize local importance and sort it
    normalized_importance = local_importance.abs() / local_importance.abs().sum()
    ordered_indices = torch.argsort(normalized_importance, descending=True)

    # print top 5 concepts
    for concept_id in ordered_indices[:5]:
        importance = normalized_importance[concept_id]
        words_importance = topk_words[concept_id.item()]
        if words_importance is not None:
            print(f"\t{concept_id}: {round(importance.item(), 3)} - {list(words_importance.keys())}")
        else:
            print(f"\t{concept_id}: {round(importance.item(), 3)} - None")

    print("\n")


Computing activations: 100%|██████████| 1/1 [00:00<00:00,  9.72batch/s]


Local importance for example of class World (pred: World):
	Sister of man who died in Vancouver police custody slams chief (Canadian Press) Canadian Press - VANCOUVER (CP) - The sister of a man who died after a violent confrontation with police has demanded the city's chief constable resign for defending the officer involved.


	11: 0.347 - ['betting', 'saudi', 'gambler', 'shark', 'kidnapper']
	27: 0.126 - ['serbia-montenegro', 'nato', 'liechtenstein', 'nikkei', 'ossetia']
	31: 0.062 - ['betting', 'fraud', 'holy', 'kmart', 'p.m.']
	6: 0.058 - ['typhoon', 'hurricane', 'earthquake', 'landslide', 'hit']
	48: 0.047 - ['afp', 'armed', 'hue', 'anarchist', 'naval']


Local importance for example of class Sports (pred: Sports):
	Giddy Phelps Touches Gold for First Time Michael Phelps won the gold medal in the 400 individual medley and set a world record in a time of 4 minutes 8.26 seconds.
	30: 0.522 - ['200-meter', '100-meter', 'breaststroke', '400-meter', 'heat']
	7: 0.124 - ['phelps', '200-meter', 'batter', 'inning', 'homered']
	35: 0.076 - ['realignment', 'dallas', 'hunger-striking', 'playboy', 'goodale']
	24: 0.072 - ['stryker', 'nfl', 'armadillo', 'homer', 'autodesk']
	16: 0.034 - ['stamps.com', 'missile-defense', 'philadelphia', 'trade', 'israeli']


Local importance for example of class Business (pred: Busine

## 7. **Evaluate** concept-based explanations <a class="anchor" id="evaluate"></a>

### 7.1 Evaluate the concept-space from the [third part](#fit)

### 7.2 Evaluate the concepts-interpretations from the [fifth step](#important)

### 7.3 Evaluate the whole concept-based explanations with `ConSim`

In [14]:
# Define the User-LLM (the meta-predictor and llm as a judge)
user_llm = OpenAILLM(api_key="YOUR_OPENAI_API_KEY", model="gpt4o-mini")

# Initialize the ConSim  with the model with split points and the user-llm
# Therefore, a given ConSim metric can be used on different explainers for cleaner comparison
con_sim = ConSim(model_with_split_points, user_llm, classes=classes)

# Select examples for evaluation
samples, labels, predictions = con_sim.select_examples(
    dataset["train"]["text"],
    dataset["train"]["label"],
)

# Compute a baseline and ConSim score to give sense to the explainer ConSim score
baseline = con_sim.evaluate(samples, labels, predictions, prompt_type=PromptTypes.L2_baseline_with_lp)

# Compute the ConSim score for an explainer # TODO: allow to give a list
con_sim_score = con_sim.evaluate(
    samples, labels, predictions, concept_explainer, prompt_type=PromptTypes.E3_global_and_local_concepts_with_lp
)

NameError: name 'OpenAILLM' is not defined

In [ ]:
activations.keys()